In [0]:
# Databricks notebook source
# Scheduled entry point for the silver layer.
#
# Rebuilds silver tables from bronze, then the link tables that join
# them. Both are pure functions of their inputs, so this is safe to run
# at any time and any number of times.

from ea_pipeline.links import build_all_link_tables
from ea_pipeline.silver import build_all_silver_tables

# COMMAND ----------

silver_summaries = build_all_silver_tables()

for summary in silver_summaries:
    print(summary)

# COMMAND ----------

# Links join silver tables, so they must run after the rebuild above.
link_summaries = build_all_link_tables()

for summary in link_summaries:
    print(summary)

# COMMAND ----------

failed = (
    [s["dataset"] for s in silver_summaries if "error" in s]
    + [s["link"] for s in link_summaries if "error" in s]
)

if failed:
    raise RuntimeError(f"Silver layer build failed for: {failed}")

dbutils.notebook.exit(str(silver_summaries + link_summaries))